# Heart Disease Classification — PyTorch vs Keras vs TensorFlow
### One Dataset, Three Frameworks — Side-by-Side Educational Guide

---

> **Goal**: Solve the **same problem** (heart disease binary classification) using three popular deep learning frameworks so you can directly compare their APIs, philosophy, and workflow.

> ✅ **CPU-friendly** — All models are small and designed to run without a GPU.

## Framework Comparison at a Glance

| Aspect | PyTorch | Keras (TF backend) | TensorFlow Low-Level |
|--------|---------|-------------------|----------------------|
| Style | Imperative / Pythonic | High-level / declarative | Explicit / flexible |
| Model definition | `nn.Module` subclass | `Sequential` or `Functional` | `tf.Module` or `GradientTape` |
| Training loop | Manual (you write it) | `model.fit()` hides it | `tf.GradientTape` manual |
| Debugging | Easy — standard Python | Harder (graph-mode) | Medium |
| Research use | Very popular | Less (PyTorch dominates) | Moderate |
| Production/deploy | TorchServe, ONNX | TF Serving, TFLite | TF Serving, TFLite |
| Best for | Research, flexibility | Rapid prototyping | Production pipelines |

## Roadmap

| Part | Framework | Topic |
|------|-----------|-------|
| 0 | All | Shared data loading & preprocessing |
| 1 | **PyTorch** | `nn.Module`, `DataLoader`, manual training loop |
| 2 | **Keras Sequential** | `model.add()`, `model.fit()`, callbacks |
| 3 | **Keras Functional API** | `Input()` → layer graph → `Model()` |
| 4 | **TensorFlow `GradientTape`** | Low-level custom training loop |
| 5 | All | Cross-framework metrics comparison |
| 6 | All | Saving & loading models (all three ways) |

---
## Part 0 — Shared Setup & Data Preprocessing

This section is **framework-agnostic**. We load, explore, clean and split the data once, then feed it into each framework.

In [ ]:
# Install all required packages (run once)
import subprocess, sys

print(f'Python version: {sys.version}')

# PyTorch + data packages — work on ALL Python versions (including 3.13+)
for p in ['ucimlrepo', 'torch', 'scikit-learn', 'matplotlib', 'seaborn', 'pandas', 'numpy']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', p, '-q'])
print('PyTorch stack installed!')

# TensorFlow — only supports Python 3.9–3.12 (not 3.13+ yet as of 2026)
if sys.version_info < (3, 13):
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tensorflow', '-q'])
        print('TensorFlow installed!')
    except Exception as e:
        print(f'TensorFlow install failed: {e}')
else:
    print()
    print('⚠️  TensorFlow NOT installed — it does not yet support Python 3.13+.')
    print('   Your Python:', sys.version.split()[0])
    print('   Parts 2, 3, 4 (Keras/TF cells) require Python 3.9–3.12.')
    print('   Part 0 & 1 (PyTorch) run fully on your current Python!')
    print()
    print('   To run TF parts, create a Python 3.12 environment:')
    print('     python3.12 -m venv .venv312 && source .venv312/bin/activate')
    print('     pip install -r requirements.txt')

print('\nDone!')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)

# ── PyTorch ───────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ── TensorFlow / Keras (optional — requires Python 3.9–3.12) ─────
# We use a try/except so the notebook doesn't crash on Python 3.13+.
# Parts 0 & 1 (PyTorch) work regardless; Parts 2–4 need TF.
TF_AVAILABLE = False
try:
    os.environ['CUDA_VISIBLE_DEVICES'] = ''    # hide GPUs → CPU only
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # suppress TF verbosity
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, callbacks
    TF_AVAILABLE = True
except ImportError:
    tf = None
    print('⚠️  TensorFlow not found. Parts 2, 3, 4 will be skipped.')
    print('   Install with: pip install tensorflow  (requires Python 3.9–3.12)')

# ── Reproducibility ───────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if TF_AVAILABLE:
    tf.random.set_seed(SEED)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
DEVICE = torch.device('cpu')   # CPU only — no GPU needed

print(f'PyTorch   : {torch.__version__}')
print(f'TF ready  : {TF_AVAILABLE}  {"("+tf.__version__+")" if TF_AVAILABLE else "(not installed)"}')
print(f'Device    : {DEVICE}')

In [ ]:
# ── Load UCI Heart Disease Dataset ────────────────────────────────
hd  = fetch_ucirepo(id=45)
X_raw = hd.data.features.copy()
y_raw = hd.data.targets.copy()

# Binarise target: 0=no disease, 1=disease
y = (y_raw['num'] > 0).astype(int)

# ── Impute missing values ─────────────────────────────────────────
for col in X_raw.select_dtypes('number').columns:
    X_raw[col].fillna(X_raw[col].median(), inplace=True)

FEATURE_NAMES = list(X_raw.columns)

# ── 70 / 15 / 15 Stratified Split ────────────────────────────────
X_np = X_raw.values.astype(np.float32)
y_np = y.values.astype(np.int64)

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_np, y_np, test_size=0.30,
                                              stratify=y_np, random_state=SEED)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50,
                                              stratify=y_tmp, random_state=SEED)

# ── Standardise (fit on train only!) ─────────────────────────────
scaler = StandardScaler()
X_tr  = scaler.fit_transform(X_tr)
X_val = scaler.transform(X_val)
X_te  = scaler.transform(X_te)

print(f'Train : {X_tr.shape[0]} | Val : {X_val.shape[0]} | Test : {X_te.shape[0]}')
print(f'Features : {X_tr.shape[1]}')

In [ ]:
# ── Quick EDA ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Class balance
pd.Series(y_np).value_counts().plot(kind='bar', ax=axes[0],
    color=['#5cb85c', '#d9534f'], edgecolor='black')
axes[0].set_xticklabels(['No Disease', 'Disease'], rotation=0)
axes[0].set_title('Class Balance')

# Feature correlation with target
df_tmp = pd.DataFrame(X_raw)
df_tmp['target'] = y_np
correlations = df_tmp.corr()['target'].drop('target').sort_values()
colors = ['#d9534f' if c > 0 else '#5cb85c' for c in correlations]
correlations.plot(kind='barh', ax=axes[1], color=colors, edgecolor='black')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Feature Correlation with Target')

# Age distribution by class
for cls, color, lbl in [(0,'#5cb85c','No Disease'),(1,'#d9534f','Disease')]:
    axes[2].hist(X_raw['age'][y_np==cls], bins=15, alpha=0.6,
                 color=color, label=lbl, edgecolor='white')
axes[2].set_title('Age Distribution by Class')
axes[2].legend()

plt.tight_layout()
plt.show()

---
# PART 1 — PyTorch
## The "Do Everything Yourself" Framework

### Philosophy
PyTorch gives you **full control**. Every step — the data pipeline, forward pass, gradient computation, weight update — is explicit Python code. This makes it:
- Easy to debug (it's just Python)
- Flexible for research (custom losses, architectures, training loops)
- More verbose than Keras

### Core PyTorch Concepts

| Concept | Purpose |
|---------|----------|
| `nn.Module` | Base class for all neural network models |
| `forward()` | Defines the computation (called on each pass) |
| `Tensor` | N-dimensional array with automatic differentiation |
| `autograd` | Tracks operations on tensors to compute gradients |
| `DataLoader` | Batches and shuffles data |
| `optimizer.zero_grad()` | Clear accumulated gradients before each step |
| `loss.backward()` | Compute gradients via backpropagation |
| `optimizer.step()` | Update weights using computed gradients |

In [ ]:
# ── PyTorch DataLoader Setup ──────────────────────────────────────
def make_loader(X, y, batch_size=32, shuffle=False):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long)
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

pt_train = make_loader(X_tr,  y_tr,  shuffle=True)
pt_val   = make_loader(X_val, y_val)
pt_test  = make_loader(X_te,  y_te)

# Inspect one batch
xb, yb = next(iter(pt_train))
print(f'Batch X: {xb.shape}  |  Batch y: {yb.shape}  |  dtype: {xb.dtype}')

In [ ]:
# ── PyTorch Model: nn.Module ──────────────────────────────────────
# KEY INSIGHT: You subclass nn.Module and implement forward().
# PyTorch builds the computation graph dynamically ("define-by-run").

class PyTorchFNN(nn.Module):
    """
    Feedforward Network: Input → [Linear→BN→ReLU→Dropout] × 2 → Output
    """
    def __init__(self, in_dim=13, hidden=[64, 32], p=0.3):
        super().__init__()
        layers_list = []
        prev = in_dim
        for h in hidden:
            layers_list += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(p)
            ]
            prev = h
        layers_list.append(nn.Linear(prev, 2))   # 2 output logits
        self.net = nn.Sequential(*layers_list)

    def forward(self, x):
        return self.net(x)     # logits (no softmax — CrossEntropyLoss handles it)


pt_model = PyTorchFNN().to(DEVICE)
print(pt_model)
n_params = sum(p.numel() for p in pt_model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {n_params:,}')

In [ ]:
# ── PyTorch Training Loop ─────────────────────────────────────────
# The training loop is EXPLICIT. Every step is visible and customisable.

pt_criterion = nn.CrossEntropyLoss()
pt_optimizer = optim.Adam(pt_model.parameters(), lr=1e-3, weight_decay=1e-4)
pt_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    pt_optimizer, mode='min', factor=0.5, patience=10, verbose=False
)

EPOCHS = 100
pt_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_pt_state, best_pt_val = None, float('inf')

for epoch in range(1, EPOCHS + 1):
    # ── TRAIN ──
    pt_model.train()   # enables Dropout + BatchNorm in train mode
    t_loss, t_correct, t_total = 0, 0, 0
    for xb, yb in pt_train:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = pt_model(xb)               # 1. Forward
        loss   = pt_criterion(logits, yb)   # 2. Loss
        pt_optimizer.zero_grad()            # 3. Clear old gradients
        loss.backward()                     # 4. Backpropagation
        pt_optimizer.step()                 # 5. Update weights
        t_loss    += loss.item() * xb.size(0)
        t_correct += (logits.argmax(1) == yb).sum().item()
        t_total   += xb.size(0)

    # ── VALIDATE ──
    pt_model.eval()    # disables Dropout, uses running stats in BatchNorm
    v_loss, v_correct, v_total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in pt_val:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = pt_model(xb)
            loss   = pt_criterion(logits, yb)
            v_loss    += loss.item() * xb.size(0)
            v_correct += (logits.argmax(1) == yb).sum().item()
            v_total   += xb.size(0)

    tl = t_loss/t_total;  ta = t_correct/t_total
    vl = v_loss/v_total;  va = v_correct/v_total
    pt_scheduler.step(vl)

    if vl < best_pt_val:
        best_pt_val   = vl
        best_pt_state = {k: v.clone() for k, v in pt_model.state_dict().items()}

    pt_history['train_loss'].append(tl);  pt_history['val_loss'].append(vl)
    pt_history['train_acc'].append(ta);   pt_history['val_acc'].append(va)

    if epoch % 20 == 0 or epoch == 1:
        print(f'Epoch {epoch:>3} | train_loss={tl:.4f} acc={ta:.3f} | val_loss={vl:.4f} acc={va:.3f}')

print('\nPyTorch training done!')

In [ ]:
# ── PyTorch: Plot Curves & Evaluate ──────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, EPOCHS+1)
ax1.plot(ep, pt_history['train_loss'], label='Train', color='steelblue')
ax1.plot(ep, pt_history['val_loss'],   label='Val',   color='tomato')
ax1.set_title('[PyTorch] Loss'); ax1.legend(); ax1.grid(alpha=0.4)
ax2.plot(ep, pt_history['train_acc'], label='Train', color='steelblue')
ax2.plot(ep, pt_history['val_acc'],   label='Val',   color='tomato')
ax2.set_title('[PyTorch] Accuracy'); ax2.legend(); ax2.grid(alpha=0.4)
plt.suptitle('PyTorch Training Curves', fontweight='bold')
plt.tight_layout(); plt.show()

# Test evaluation
pt_model.load_state_dict(best_pt_state)
pt_model.eval()
all_probs, all_preds, all_labels = [], [], []
with torch.no_grad():
    for xb, yb in pt_test:
        logits = pt_model(xb.to(DEVICE))
        probs  = torch.softmax(logits, 1)[:, 1].cpu().numpy()
        preds  = logits.argmax(1).cpu().numpy()
        all_probs.extend(probs); all_preds.extend(preds)
        all_labels.extend(yb.numpy())

pt_results = {
    'acc':  accuracy_score(all_labels, all_preds),
    'f1':   f1_score(all_labels, all_preds),
    'auc':  roc_auc_score(all_labels, all_probs)
}
print(f"[PyTorch] Acc={pt_results['acc']:.4f}  F1={pt_results['f1']:.4f}  AUC={pt_results['auc']:.4f}")

---
# PART 2 — Keras Sequential API
## The "Stack Layers" High-Level Framework

### Philosophy
Keras (with TensorFlow backend) is designed for **rapid prototyping**. You describe *what* you want, and Keras handles *how* to do it.

The **Sequential API** stacks layers linearly — perfect for standard feedforward networks.

### Key Concepts

| PyTorch equivalent | Keras equivalent | Notes |
|--------------------|-----------------|-------|
| `nn.Module` subclass | `keras.Sequential` | Keras model container |
| `nn.Linear(in, out)` | `layers.Dense(out)` | Keras infers input shape |
| `nn.BatchNorm1d(h)` | `layers.BatchNormalization()` | Same effect |
| `nn.ReLU()` | `layers.Activation('relu')` or `activation='relu'` | |
| `nn.Dropout(p)` | `layers.Dropout(p)` | Same |
| Manual training loop | `model.fit(X, y, ...)` | Keras handles everything |
| `ReduceLROnPlateau` | `callbacks.ReduceLROnPlateau` | Same concept, as a callback |
| Manual early stopping | `callbacks.EarlyStopping` | Built-in to Keras |

### Callbacks
Instead of writing logic inside a training loop, Keras uses **callbacks** — hooks that fire at specific points (epoch start/end, batch start/end). Common ones:
- `EarlyStopping` — stop when val_loss stops improving
- `ReduceLROnPlateau` — reduce LR on plateau
- `ModelCheckpoint` — save best model automatically

In [ ]:
if not TF_AVAILABLE:
    print("⚠️  Skipping Part 2 — TensorFlow not installed.")
    print("   Upgrade to Python 3.12 and run: pip install tensorflow")
else:
    # ── Keras Sequential API ──────────────────────────────────────────
    # NOTICE: We call model.compile() to configure loss/optimizer/metrics
    # then model.fit() — no explicit loop needed!

    def build_sequential_model(input_dim=13, hidden=[64, 32], dropout=0.3):
        """
        Build a Feedforward Neural Network using Keras Sequential API.
        Same architecture as PyTorchFNN for a fair comparison.
        """
        model = keras.Sequential(name='SequentialFNN')
        model.add(keras.Input(shape=(input_dim,)))   # explicit input shape

        for h in hidden:
            model.add(layers.Dense(h))               # Linear transformation
            model.add(layers.BatchNormalization())    # Normalise activations
            model.add(layers.Activation('relu'))      # Non-linearity
            model.add(layers.Dropout(dropout))        # Regularisation

        model.add(layers.Dense(1, activation='sigmoid'))  # Binary output
        return model

    keras_seq = build_sequential_model()

    # compile() sets the loss, optimizer, and metrics to track
    keras_seq.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',    # sigmoid output → binary cross-entropy
        metrics=['accuracy']
    )
    keras_seq.summary()

In [ ]:
if not TF_AVAILABLE:
    print("⚠️  Skipping — TensorFlow not installed.")
else:
    # ── Keras Callbacks ───────────────────────────────────────────────
    keras_callbacks = [
        callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                 restore_best_weights=True, verbose=0),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                     patience=10, verbose=0)
    ]

    # ── model.fit() — Keras handles the training loop internally ──────
    keras_history = keras_seq.fit(
        X_tr,  y_tr.astype('float32'),
        validation_data=(X_val, y_val.astype('float32')),
        epochs=150, batch_size=32,
        callbacks=keras_callbacks, verbose=0
    )
    print(f'Stopped at epoch: {len(keras_history.history["loss"])}')

In [ ]:
if not TF_AVAILABLE:
    print("⚠️  Skipping — TensorFlow not installed.")
    ks_results = None
else:
    h = keras_history.history
    ep_range = range(1, len(h['loss'])+1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ax1.plot(ep_range, h['loss'],     label='Train', color='steelblue')
    ax1.plot(ep_range, h['val_loss'], label='Val',   color='tomato')
    ax1.set_title('[Keras Sequential] Loss'); ax1.legend(); ax1.grid(alpha=0.4)
    ax2.plot(ep_range, h['accuracy'],     label='Train', color='steelblue')
    ax2.plot(ep_range, h['val_accuracy'], label='Val',   color='tomato')
    ax2.set_title('[Keras Sequential] Accuracy'); ax2.legend(); ax2.grid(alpha=0.4)
    plt.suptitle('Keras Sequential Training Curves', fontweight='bold')
    plt.tight_layout(); plt.show()

    ks_probs = keras_seq.predict(X_te, verbose=0).flatten()
    ks_preds = (ks_probs >= 0.5).astype(int)
    ks_results = {
        'acc': accuracy_score(y_te, ks_preds),
        'f1':  f1_score(y_te, ks_preds),
        'auc': roc_auc_score(y_te, ks_probs)
    }
    print(f"[Keras Sequential] Acc={ks_results['acc']:.4f}  F1={ks_results['f1']:.4f}  AUC={ks_results['auc']:.4f}")

---
# PART 3 — Keras Functional API
## The "Flexible Graph" Approach

### Why Functional API?

Sequential works only for **linear stacks** of layers. The **Functional API** represents models as a **directed acyclic graph (DAG)** of layers.

Use it when you need:
- **Multiple inputs** (e.g., image + metadata together)
- **Multiple outputs** (e.g., multi-task learning)
- **Skip connections / Residual blocks** (like ResNet)
- **Branching paths** in the network

### Syntax Difference

```python
# Sequential:
model = keras.Sequential([Dense(64), ReLU(), Dense(2)])

# Functional:
inputs = keras.Input(shape=(13,))
x = Dense(64)(inputs)    # call layer as a function on a tensor
x = ReLU()(x)
outputs = Dense(2)(x)
model = keras.Model(inputs=inputs, outputs=outputs)
```

The key insight: **layers are called as functions**, and you connect them explicitly.

In [ ]:
if not TF_AVAILABLE:
    print("⚠️  Skipping Part 3 — TensorFlow not installed.")
else:
    # ── Keras Functional API ──────────────────────────────────────────
    def build_functional_model(input_dim=13, hidden=[64, 32], dropout=0.3):
        """
        Same architecture as Sequential, built with Functional API.
        Advantage: explicit tensor flow makes it easy to add skip connections.
        """
        inputs = keras.Input(shape=(input_dim,), name='clinical_features')
        x = inputs
        for i, h in enumerate(hidden):
            x = layers.Dense(h, name=f'dense_{i+1}')(x)
            x = layers.BatchNormalization(name=f'bn_{i+1}')(x)
            x = layers.Activation('relu', name=f'relu_{i+1}')(x)
            x = layers.Dropout(dropout, name=f'dropout_{i+1}')(x)
        outputs = layers.Dense(1, activation='sigmoid', name='output')(x)
        model = keras.Model(inputs=inputs, outputs=outputs, name='FunctionalFNN')
        return model

    keras_func = build_functional_model()
    keras_func.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy', metrics=['accuracy']
    )
    keras_func.summary()

In [ ]:
if not TF_AVAILABLE:
    print("⚠️  Skipping — TensorFlow not installed.")
else:
    # ── Demo: Residual / Skip Connection with Functional API ──────────
    # This is IMPOSSIBLE with Sequential — shows the power of Functional API.
    def build_residual_model(input_dim=13, hidden_dim=64, dropout=0.3):
        """
        Network with a skip (residual) connection.
        The identity shortcut allows gradients to flow directly through.
        """
        inputs   = keras.Input(shape=(input_dim,), name='input')
        x  = layers.Dense(hidden_dim)(inputs)
        x  = layers.BatchNormalization()(x)
        x  = layers.Activation('relu')(x)
        x  = layers.Dropout(dropout)(x)
        x  = layers.Dense(hidden_dim)(x)
        x  = layers.BatchNormalization()(x)
        # Skip connection — project input to same dim
        shortcut = layers.Dense(hidden_dim, use_bias=False)(inputs)
        x = layers.Add()([x, shortcut])
        x = layers.Activation('relu')(x)
        x = layers.Dropout(dropout)(x)
        z = layers.Dense(32)(x)
        z = layers.BatchNormalization()(z)
        z = layers.Activation('relu')(z)
        outputs = layers.Dense(1, activation='sigmoid')(z)
        return keras.Model(inputs, outputs, name='ResidualFNN')

    keras_res = build_residual_model()
    keras_res.compile(optimizer=keras.optimizers.Adam(1e-3),
                      loss='binary_crossentropy', metrics=['accuracy'])
    print('Residual model parameters:', keras_res.count_params())
    keras_res.summary()

In [ ]:
if not TF_AVAILABLE:
    print("⚠️  Skipping — TensorFlow not installed.")
else:
    callbacks_func = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                      restore_best_weights=True, verbose=0),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=10, verbose=0),
    ]

    print("Training Keras Functional model …")
    kf_hist = keras_func.fit(
        X_train_np, y_train_np,
        validation_data=(X_val_np, y_val_np),
        epochs=200, batch_size=32,
        callbacks=callbacks_func, verbose=0)

    print("Training Keras Residual model …")
    kr_hist = keras_res.fit(
        X_train_np, y_train_np,
        validation_data=(X_val_np, y_val_np),
        epochs=200, batch_size=32,
        callbacks=callbacks_func, verbose=0)

    print("Done training functional models.")

In [ ]:
if not TF_AVAILABLE:
    print("⚠️  Skipping — TensorFlow not installed.")
    kf_results, kr_results = None, None
else:
    def keras_evaluate(model, hist, name, X_te, y_te):
        preds = (model.predict(X_te, verbose=0) > 0.5).astype(int).ravel()
        probs = model.predict(X_te, verbose=0).ravel()
        acc  = accuracy_score(y_te, preds)
        f1   = f1_score(y_te, preds)
        roc  = roc_auc_score(y_te, probs)
        print(f"\n{name}  Acc={acc:.4f}  F1={f1:.4f}  AUC={roc:.4f}")
        print(classification_report(y_te, preds,
                                    target_names=['No Disease','Disease']))
        return dict(name=name, accuracy=acc, f1=f1, roc_auc=roc,
                    probs=probs, y_true=y_te,
                    train_loss=hist.history['loss'],
                    val_loss=hist.history['val_loss'])

    kf_results = keras_evaluate(keras_func, kf_hist,
                                'Keras Functional', X_test_np, y_test_np)
    kr_results = keras_evaluate(keras_res, kr_hist,
                                'Keras Residual',   X_test_np, y_test_np)

    # ── Loss curves ──────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, hist, title in zip(axes,
                                [kf_hist, kr_hist],
                                ['Keras Functional', 'Keras Residual']):
        ax.plot(hist.history['loss'],     label='Train loss')
        ax.plot(hist.history['val_loss'], label='Val loss')
        ax.set_title(f'{title} — Loss curves')
        ax.set_xlabel('Epoch'); ax.set_ylabel('BCE Loss')
        ax.legend(); ax.grid(True)
    plt.tight_layout(); plt.show()

---
# PART 4 — TensorFlow Low-Level: `GradientTape`
## The "Glass Box" Manual Training Loop

### Why Learn This?

`tf.GradientTape` is TensorFlow's equivalent of PyTorch's `loss.backward()`.  
It gives you **full control** over gradient computation — essential when:
- Implementing **custom training objectives** (e.g., GANs, meta-learning)
- **Multi-loss** training (e.g., classification + reconstruction)
- Debugging gradient flow
- Research requiring non-standard training procedures

### How `GradientTape` Works

```python
with tf.GradientTape() as tape:
    predictions = model(X, training=True)   # forward pass (tape records ops)
    loss = loss_fn(y_true, predictions)     # compute loss

# tape.gradient() computes ∂loss/∂each_trainable_variable
grads = tape.gradient(loss, model.trainable_variables)

# Apply gradients (weight update)
optimizer.apply_gradients(zip(grads, model.trainable_variables))
```

By default, TF uses a **static computation graph** (`@tf.function` decorator)  
for faster execution — or **eager mode** for easier debugging.

In [ ]:
if not TF_AVAILABLE:
    print("⚠️  Skipping Part 4 — TensorFlow not installed.")
else:
    # ── Build model & tf.data.Dataset ─────────────────────────────────
    tf_model = build_functional_model()   # reuse the same architecture
    tf_model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy', metrics=['accuracy'])

    # tf.data pipeline (shuffle → batch → prefetch)
    BATCH  = 32
    BUFFER = len(X_train_np)

    train_ds = (tf.data.Dataset
                .from_tensor_slices((X_train_np.astype('float32'),
                                     y_train_np.astype('float32')))
                .shuffle(BUFFER, seed=42)
                .batch(BATCH)
                .prefetch(tf.data.AUTOTUNE))

    val_ds   = (tf.data.Dataset
                .from_tensor_slices((X_val_np.astype('float32'),
                                     y_val_np.astype('float32')))
                .batch(BATCH)
                .prefetch(tf.data.AUTOTUNE))

    test_ds  = (tf.data.Dataset
                .from_tensor_slices((X_test_np.astype('float32'),
                                     y_test_np.astype('float32')))
                .batch(BATCH))

    print("tf.data.Dataset ready — tf_model parameters:", tf_model.count_params())

In [ ]:
if not TF_AVAILABLE:
    print("⚠️  Skipping — TensorFlow not installed.")
else:
    loss_fn   = keras.losses.BinaryCrossentropy()
    optimizer = keras.optimizers.Adam(1e-3)

    @tf.function
    def train_step(x_batch, y_batch):
        with tf.GradientTape() as tape:
            logits = tf_model(x_batch, training=True)
            loss   = loss_fn(y_batch[:, None], logits)
        grads = tape.gradient(loss, tf_model.trainable_variables)
        optimizer.apply_gradients(zip(grads, tf_model.trainable_variables))
        return loss

    @tf.function
    def val_step(x_batch, y_batch):
        logits = tf_model(x_batch, training=False)
        return loss_fn(y_batch[:, None], logits)

    EPOCHS      = 200
    PATIENCE    = 20
    best_val    = float('inf')
    wait        = 0
    best_weights= None
    tf_train_loss, tf_val_loss = [], []

    for epoch in range(EPOCHS):
        # Training
        batch_losses = [train_step(xb, yb)
                        for xb, yb in train_ds]
        epoch_train  = float(tf.reduce_mean(batch_losses))

        # Validation
        val_losses   = [val_step(xb, yb) for xb, yb in val_ds]
        epoch_val    = float(tf.reduce_mean(val_losses))

        tf_train_loss.append(epoch_train)
        tf_val_loss.append(epoch_val)

        # Early stopping
        if epoch_val < best_val - 1e-4:
            best_val     = epoch_val
            wait         = 0
            best_weights = tf_model.get_weights()
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"Early stop at epoch {epoch+1}")
                break

    tf_model.set_weights(best_weights)
    print(f"Training complete — best val_loss: {best_val:.4f}")

In [ ]:
if not TF_AVAILABLE:
    print("⚠️  Skipping — TensorFlow not installed.")
    tf_results = None
else:
    tf_probs = tf_model.predict(X_test_np.astype('float32'), verbose=0).ravel()
    tf_preds = (tf_probs > 0.5).astype(int)
    tf_acc   = accuracy_score(y_test_np, tf_preds)
    tf_f1    = f1_score(y_test_np, tf_preds)
    tf_roc   = roc_auc_score(y_test_np, tf_probs)
    print(f"\nTF GradientTape  Acc={tf_acc:.4f}  F1={tf_f1:.4f}  AUC={tf_roc:.4f}")
    print(classification_report(y_test_np, tf_preds,
                                 target_names=['No Disease','Disease']))

    tf_results = dict(name='TF GradientTape', accuracy=tf_acc,
                      f1=tf_f1, roc_auc=tf_roc,
                      probs=tf_probs, y_true=y_test_np,
                      train_loss=tf_train_loss,
                      val_loss=tf_val_loss)

    # Loss curve
    plt.figure(figsize=(7, 4))
    plt.plot(tf_train_loss, label='Train loss')
    plt.plot(tf_val_loss,   label='Val loss')
    plt.title('TF GradientTape — Loss curves')
    plt.xlabel('Epoch'); plt.ylabel('BCE Loss')
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

---
# PART 5 — Cross-Framework Comparison

Now that all models are trained, let's compare them head-to-head.

In [ ]:
# ── Collect only the frameworks that were actually run ───────────────
_all = {
    'PyTorch FNN':    pt_results,
    'Keras Seq':      ks_results,
    'Keras Func':     kf_results,
    'Keras Residual': kr_results,
    'TF GradTape':    tf_results,
}
available = {k: v for k, v in _all.items() if v is not None}

if not available:
    print("No results to compare — only PyTorch was available on this Python version.")
else:
    rows = []
    for name, r in available.items():
        rows.append({
            'Framework': name,
            'Accuracy':  round(r['accuracy'], 4),
            'F1-Score':  round(r['f1'],       4),
            'ROC-AUC':   round(r['roc_auc'],  4),
        })
    df_cmp = pd.DataFrame(rows).set_index('Framework')
    print("=" * 55)
    print("       FRAMEWORK COMPARISON")
    print("=" * 55)
    print(df_cmp.to_string())
    print("=" * 55)
    print("\nBest accuracy :", df_cmp['Accuracy'].idxmax())
    print("Best F1-Score  :", df_cmp['F1-Score'].idxmax())
    print("Best ROC-AUC   :", df_cmp['ROC-AUC'].idxmax())

# Store for downstream cells
if 'df_cmp' not in dir():
    df_cmp = None

In [ ]:
if df_cmp is None:
    print("⚠️  No comparison data available — skipping bar chart.")
else:
    ax = df_cmp[['Accuracy', 'F1-Score', 'ROC-AUC']].plot(
        kind='bar', figsize=(10, 5), rot=30,
        color=['steelblue', 'coral', 'seagreen'], alpha=0.85, edgecolor='black')
    ax.set_title('Framework Comparison — Test-Set Metrics', fontsize=14)
    ax.set_ylabel('Score'); ax.set_ylim(0.5, 1.05)
    ax.axhline(1.0, color='grey', lw=0.8, ls='--')
    ax.legend(loc='upper left')
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.3f}',
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=8, rotation=0)
    plt.tight_layout(); plt.show()

In [ ]:
_all_roc = {
    'PyTorch FNN':    pt_results,
    'Keras Seq':      ks_results,
    'Keras Func':     kf_results,
    'Keras Residual': kr_results,
    'TF GradTape':    tf_results,
}
available_roc = {k: v for k, v in _all_roc.items() if v is not None}

if not available_roc:
    print("⚠️  No ROC data available.")
else:
    from sklearn.metrics import roc_curve
    plt.figure(figsize=(8, 6))
    colors = ['royalblue', 'tomato', 'seagreen', 'darkorange', 'purple']
    for (name, r), color in zip(available_roc.items(), colors):
        fpr, tpr, _ = roc_curve(r['y_true'], r['probs'])
        plt.plot(fpr, tpr, lw=2, color=color,
                 label=f"{name} (AUC={r['roc_auc']:.3f})")
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title('ROC Curves — All Available Frameworks')
    plt.legend(loc='lower right'); plt.grid(True)
    plt.tight_layout(); plt.show()

---
# PART 6 — Saving & Loading Models

Every framework has its own model persistence format. Knowing how to save and reload models is critical for deployment.

| Framework | Native format | Interop format |
|-----------|--------------|----------------|
| PyTorch | `.pt` / `.pth` (state dict) | ONNX |
| Keras/TF | `.keras` (new) / `SavedModel` dir | ONNX, TFLite |
| All | ONNX | ONNX (universal) |

In [ ]:
import os
save_dir = 'saved_models'
os.makedirs(save_dir, exist_ok=True)

# ── PyTorch (always available) ────────────────────────────────────────
torch.save(pt_model.state_dict(), f'{save_dir}/pytorch_fnn.pth')
print(f"✅  PyTorch model saved → {save_dir}/pytorch_fnn.pth")

# ── Keras / TF (only if TF available) ────────────────────────────────
if not TF_AVAILABLE:
    print("⚠️  Skipping Keras/TF model saving — TensorFlow not installed.")
else:
    keras_seq.save(f'{save_dir}/keras_sequential.keras')
    keras_func.save(f'{save_dir}/keras_functional.keras')
    keras_res.save(f'{save_dir}/keras_residual.keras')
    tf_model.save(f'{save_dir}/tf_gradienttape.keras')
    print(f"✅  Keras Sequential  saved → {save_dir}/keras_sequential.keras")
    print(f"✅  Keras Functional  saved → {save_dir}/keras_functional.keras")
    print(f"✅  Keras Residual    saved → {save_dir}/keras_residual.keras")
    print(f"✅  TF GradientTape   saved → {save_dir}/tf_gradienttape.keras")

In [ ]:
# ── PyTorch reload ────────────────────────────────────────────────────
loaded_pt = HeartDiseaseNet(input_dim=X_train_t.shape[1])
loaded_pt.load_state_dict(torch.load(f'{save_dir}/pytorch_fnn.pth',
                                     map_location='cpu'))
loaded_pt.eval()
with torch.no_grad():
    sample = torch.FloatTensor(X_test_t[:5])
    preds  = (loaded_pt(sample) > 0.5).int().numpy().ravel()
print("PyTorch reloaded predictions (first 5):", preds)
print("True labels (first 5):                 ", y_test_t[:5].numpy())

# ── Keras / TF reload ────────────────────────────────────────────────
if not TF_AVAILABLE:
    print("\n⚠️  Skipping Keras/TF model loading — TensorFlow not installed.")
else:
    loaded_ks = keras.models.load_model(f'{save_dir}/keras_sequential.keras')
    loaded_kf = keras.models.load_model(f'{save_dir}/keras_functional.keras')
    loaded_kr = keras.models.load_model(f'{save_dir}/keras_residual.keras')
    loaded_tf = keras.models.load_model(f'{save_dir}/tf_gradienttape.keras')
    # Quick sanity check
    sample_np = X_test_np[:5].astype('float32')
    for name, m in [('Keras Sequential', loaded_ks),
                    ('Keras Functional', loaded_kf),
                    ('Keras Residual',   loaded_kr),
                    ('TF GradientTape',  loaded_tf)]:
        p = (m.predict(sample_np, verbose=0) > 0.5).astype(int).ravel()
        print(f"{name} reloaded predictions (first 5): {p}")

---
# PART 7 — Framework Philosophy Summary

## When to Choose Which Framework?

```
┌─────────────────────────────────────────────────────────────────────┐
│                        YOUR GOAL                                    │
│                                                                     │
│  Research / Custom      →   PyTorch                                 │
│  architecture research       • Full control, Pythonic               │
│  (papers, experiments)       • Easy debugging                       │
│                              • Dynamic computation graph            │
│                                                                     │
│  Rapid prototyping       →   Keras Sequential                       │
│  (prove concept fast)        • Minimal code                         │
│                              • model.fit() handles everything       │
│                              • Great for standard architectures     │
│                                                                     │
│  Complex architectures   →   Keras Functional                       │
│  (multi-input/output,        • Skip connections (ResNet)            │
│   skip connections)          • Multi-task learning                  │
│                              • Still high-level                     │
│                                                                     │
│  Custom training logic   →   TF GradientTape                        │
│  (GANs, meta-learning,       • Fine-grained gradient control        │
│   custom losses)             • Works within TF ecosystem            │
│                              • good for TF Serving production       │
└─────────────────────────────────────────────────────────────────────┘
```

## API Cheatsheet — The Same 5 Operations

| Operation | PyTorch | Keras/TF |
|-----------|---------|----------|
| **Define model** | `class Net(nn.Module): def forward(x)` | `Sequential()` / `keras.Model(inputs, outputs)` |
| **Linear layer** | `nn.Linear(in, out)` | `layers.Dense(out)` |
| **Activation** | `nn.ReLU()` | `layers.Activation('relu')` |
| **Dropout** | `nn.Dropout(p)` | `layers.Dropout(p)` |
| **BatchNorm** | `nn.BatchNorm1d(h)` | `layers.BatchNormalization()` |
| **Loss (binary)** | `nn.BCELoss()` or `CrossEntropyLoss` | `binary_crossentropy` |
| **Optimizer** | `optim.Adam(model.parameters(), lr=...)` | `keras.optimizers.Adam(lr=...)` |
| **Train mode** | `model.train()` | `training=True` in call |
| **Eval mode** | `model.eval()` | `training=False` in call |
| **No gradient** | `with torch.no_grad():` | `@tf.function` / no tape |
| **Save weights** | `torch.save(model.state_dict(), path)` | `model.save_weights(path)` |
| **Load weights** | `model.load_state_dict(torch.load(path))` | `model.load_weights(path)` |
| **LR scheduler** | `optim.lr_scheduler.*` | `callbacks.ReduceLROnPlateau` |
| **Early stopping** | Manual in training loop | `callbacks.EarlyStopping` |
| **Data pipeline** | `DataLoader(TensorDataset(...))` | `tf.data.Dataset.from_tensor_slices(...)` |

In [ ]:
print("=" * 60)
print("  MULTI-FRAMEWORK DEEP LEARNING TUTORIAL — COMPLETE")
print("=" * 60)

print("\n✅  PART 1  — PyTorch FNN with manual training loop")
print(f"   Accuracy: {pt_results['accuracy']:.4f}  |  "
      f"F1: {pt_results['f1']:.4f}  |  AUC: {pt_results['roc_auc']:.4f}")

if TF_AVAILABLE:
    for label, r in [('Keras Sequential', ks_results),
                     ('Keras Functional', kf_results),
                     ('Keras Residual',   kr_results),
                     ('TF GradientTape',  tf_results)]:
        if r:
            print(f"\n✅  {label}")
            print(f"   Accuracy: {r['accuracy']:.4f}  |  "
                  f"F1: {r['f1']:.4f}  |  AUC: {r['roc_auc']:.4f}")
else:
    print("\n⚠️  Parts 2–4 (Keras & TensorFlow) were skipped.")
    print("   TF requires Python < 3.13. Current env: Python 3.14.")
    print("   Upgrade to a Python 3.10–3.12 virtual environment")
    print("   and re-run to see all framework comparisons.")

print("\n" + "=" * 60)
print("KEY TAKEAWAYS")
print("=" * 60)
print("""
PyTorch   — Explicit control over every operation; ideal for research
            and custom training loops.
Keras Seq — Fastest to prototype; great for simple stacked architectures.
Keras Func— Enables multi-input/output and skip connections; more flexible.
TF Tape   — Surgical gradient control for meta-learning, adversarial
            training, and research-grade customisation.

All frameworks can represent the same mathematical model.
Choice depends on: control needed, deployment target, team preference.
""")